# NB7 - Inference latency and memory
Measures per-image inference time, FPS, and peak GPU memory for YOLOv8s, RT-DETR-l, and YOLO-World. Answers R1-W3. Self-contained and does not need the lost fine-tuned weights: these are architecture-level costs, measured from the official pretrained checkpoints.

## 0. One-time environment setup

In [ ]:
# Same pins as the other notebooks. Uninstall ray to avoid the broken raytune callback.
! pip install -q ultralytics==8.3.40 pillow==10.4.0
! pip uninstall -y -q ray 2>/dev/null || true

## 1. Imports, constants, config

In [ ]:
# Standard library
import json, os, sys, time, subprocess
from pathlib import Path

# Block ultralytics 8.3.40's broken raytune callback
sys.modules['ray'] = None

# Numeric and plotting
import numpy as np
import pandas as pd

# Deep learning
import torch

# Reproducibility
SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)

# Paths (Kaggle)
KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT   = Path('/kaggle/input')
RESULTS_DIR    = KAGGLE_WORKING / 'results'
SAMPLE_DIR     = KAGGLE_WORKING / 'latency_samples'
RESULTS_DIR.mkdir(parents=True, exist_ok=True); SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# Benchmark config
IMGSZ     = 640
BATCH     = 1            # latency reported per image
N_WARMUP  = 10           # discarded warm-up iterations (autotune, weight load)
N_MEASURE = 100          # timed iterations
DEVICE    = 0
CONF      = 0.25

# Models: architecture inference cost from the official pretrained weights.
# The fine-tuned container / ICDAR checkpoints are not needed here: latency and peak
# memory are properties of the architecture and input size, not the learned weights.
MODELS = [
    ('yolov8s',    'yolo',   'yolov8s.pt',         None),
    ('rtdetr-l',   'rtdetr', 'rtdetr-l.pt',        None),
    ('yolo-world', 'world',  'yolov8s-worldv2.pt', ['text']),
]

DATASET_SLUG = 'ausdm-v3-latency'
KAGGLE_USERNAME_OVERRIDE = ''

print('device        :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('imgsz / batch :', IMGSZ, BATCH)
print('warmup/measure:', N_WARMUP, N_MEASURE)

## 2. Neutralise the raytune callback

In [ ]:
# Neutralise ultralytics 8.3.40's broken raytune callback (same patch as the other notebooks)
try:
    from ultralytics.utils.callbacks import raytune as _rt
    _rt.on_fit_epoch_end.__code__ = (lambda trainer: None).__code__
    _rt.callbacks = {}; _rt.tune = None
    print('raytune callback neutralised')
except Exception as e:
    print('raytune patch skipped:', e)

## 3. Sample images to time on

In [ ]:
# _collect_images - gather N images to time on, self-contained with fallbacks.
# Order: ICDAR raw test images if mounted, any jpg under /kaggle/input, else synthetic
# images made in memory. Synthetic images stay as numpy arrays and are never written to
# disk, which sidesteps Kaggle's broken PIL JPEG encoder. Image content does not affect
# FPS materially. predict() accepts file paths and np arrays alike.
def _collect_images(n):
    items, src = [], 'synthetic (in-memory)'
    if KAGGLE_INPUT.exists():
        for hit in KAGGLE_INPUT.rglob('ch4_test_images'):
            if hit.is_dir():
                cand = sorted(str(p) for p in hit.rglob('*.jpg'))
                if cand:
                    items, src = cand, 'ICDAR test'
                    break
        if not items:
            found = [str(p) for p in KAGGLE_INPUT.rglob('*.jpg')]
            if found:
                items, src = found, 'input jpg'
    if len(items) < n:
        need = n - len(items)
        src = 'synthetic (in-memory)' if not items else src + '+synthetic'
        rng = np.random.default_rng(SEED)
        items = items + [rng.integers(0, 256, (720, 1280, 3), dtype=np.uint8) for _ in range(need)]
    items = items[:n]
    print(f'collected {len(items)} images (source: {src})')
    return items

IMAGES = _collect_images(N_MEASURE)

## 4. Benchmark function

In [ ]:
# _load_model / _bench - per-image inference latency and peak GPU memory.
# Latency uses ultralytics' own 'inference' timing (pure forward pass, ms/img), averaged
# over N_MEASURE images after N_WARMUP discarded iterations. Peak memory is the CUDA
# max_memory_allocated over the timed run.
def _load_model(kind, weights, classes):
    if kind == 'yolo':
        from ultralytics import YOLO; m = YOLO(weights)
    elif kind == 'rtdetr':
        from ultralytics import RTDETR; m = RTDETR(weights)
    else:
        from ultralytics import YOLOWorld; m = YOLOWorld(weights)
        if classes: m.set_classes(classes)
    return m

def _params_m(m):
    try:
        return round(sum(p.numel() for p in m.model.parameters()) / 1e6, 2)
    except Exception:
        return None

def _bench(name, kind, weights, classes):
    m = _load_model(kind, weights, classes)
    for i in range(N_WARMUP):
        m.predict(IMAGES[i % len(IMAGES)], imgsz=IMGSZ, device=DEVICE, conf=CONF, verbose=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(DEVICE)
    inf_ms, pre_ms, post_ms, wall = [], [], [], []
    for i in range(N_MEASURE):
        t0 = time.perf_counter()
        res = m.predict(IMAGES[i % len(IMAGES)], imgsz=IMGSZ, device=DEVICE, conf=CONF, verbose=False)[0]
        if torch.cuda.is_available(): torch.cuda.synchronize()
        wall.append((time.perf_counter() - t0) * 1000)
        sp = res.speed
        pre_ms.append(sp.get('preprocess', 0.0)); inf_ms.append(sp.get('inference', 0.0))
        post_ms.append(sp.get('postprocess', 0.0))
    peak = torch.cuda.max_memory_allocated(DEVICE) / 1e6 if torch.cuda.is_available() else None
    inf = np.array(inf_ms)
    row = {'model': name, 'params_M': _params_m(m),
           'inference_ms_mean': round(float(inf.mean()), 2),
           'inference_ms_std':  round(float(inf.std()), 2),
           'fps': round(1000.0 / float(inf.mean()), 1) if inf.mean() > 0 else None,
           'preprocess_ms_mean':  round(float(np.mean(pre_ms)), 2),
           'postprocess_ms_mean': round(float(np.mean(post_ms)), 2),
           'end2end_ms_mean': round(float(np.mean(wall)), 2),
           'peak_gpu_mem_MB': round(float(peak), 1) if peak is not None else None}
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return row

## 5. Run the benchmark for all three models

In [ ]:
rows = []
for name, kind, weights, classes in MODELS:
    print(f'benchmarking {name} ...')
    rows.append(_bench(name, kind, weights, classes))

df_lat = pd.DataFrame(rows)[['model','params_M','inference_ms_mean','inference_ms_std','fps',
                             'preprocess_ms_mean','postprocess_ms_mean','end2end_ms_mean','peak_gpu_mem_MB']]
df_lat.to_csv(RESULTS_DIR / 'latency_memory.csv', index=False)
print()
print(df_lat.to_string(index=False))

**5.1 Explanation.**
- `inference_ms_mean` is the pure forward-pass time per image; `fps` is its reciprocal. `end2end_ms_mean` adds preprocess and postprocess, i.e. what a user waits per image.
- `peak_gpu_mem_MB` is the CUDA peak during inference, the memory figure Reviewer #1 asked for.
- Expect YOLOv8s fastest and lightest, RT-DETR-l slowest and heaviest (transformer decoder), YOLO-World between the two. Read the printed table.
- All three are timed on the same GPU in the same session, so the columns are directly comparable.

## 6. Optional release of the CSV with read-back

In [ ]:
# Optional release of the CSV as a small dataset, with the same read-back discipline.
# The CSV is already captured inline above, so this push is a convenience, not the record.
def _kaggle_user():
    if KAGGLE_USERNAME_OVERRIDE: return KAGGLE_USERNAME_OVERRIDE
    try:
        return json.loads((Path.home()/'.kaggle'/'kaggle.json').read_text())['username']
    except Exception:
        return os.environ.get('KAGGLE_USERNAME', 'unknown')

KAGGLE_USER = _kaggle_user()
PUSH_DIR = KAGGLE_WORKING / 'lat_push'

def _push_and_verify():
    import shutil
    if KAGGLE_USER == 'unknown':
        print('Kaggle username unknown; skipping push. CSV saved inline and in', RESULTS_DIR)
        return
    if PUSH_DIR.exists(): shutil.rmtree(PUSH_DIR)
    PUSH_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(RESULTS_DIR / 'latency_memory.csv', PUSH_DIR / 'latency_memory.csv')
    (PUSH_DIR / 'dataset-metadata.json').write_text(json.dumps(
        {'title':'AusDM v3 latency and memory', 'id':f'{KAGGLE_USER}/{DATASET_SLUG}',
         'licenses':[{'name':'CC-BY-SA-4.0'}]}, indent=2))
    def _run(sub):
        r = subprocess.run(['kaggle','datasets']+sub, capture_output=True, text=True)
        return (r.stdout + r.stderr)
    out = _run(['create','-p',str(PUSH_DIR),'-r','zip'])
    if not ('is being created' in out.lower() or 'successfully' in out.lower()):
        out = _run(['version','-p',str(PUSH_DIR),'-m','NB7 latency','-r','zip'])
    print('push:', out.strip()[-200:])
    time.sleep(15)
    chk = _run(['datasets','files','-v', f'{KAGGLE_USER}/{DATASET_SLUG}'])
    print('READ-BACK VERIFY:', 'PASS' if 'latency_memory.csv' in chk else 'FAIL (CSV still saved inline and in results/)')

_push_and_verify()

## 7. Conclusion

In [ ]:
print('=== NB7 summary (per-image, imgsz 640, batch 1) ===')
print(pd.read_csv(RESULTS_DIR / 'latency_memory.csv').to_string(index=False))
print()
print('GPU        :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Note       : measured on the official pretrained architectures, not the fine-tuned')
print('             checkpoints. Latency and peak memory depend on architecture and input')
print('             size, so these are the numbers to report for R1-W3 (inference cost).')
print('Next in NB5: add fps and peak_gpu_mem_MB as columns to Table 1.')